In [7]:
# =============================================================================
# Custom SLM — Kaggle 2x Tesla T4 (2x16 GB VRAM) — Train from Scratch
#
# Kaggle Settings (right-hand panel) before running:
#   Accelerator  → GPU T4 x2
#   Internet     → On
#
# Memory budget per GPU (16 GB each, total 32 GB):
#   Model weights  fp16   ~3 GB for 1.3 B params
#   Gradients      fp16   ~3 GB
#   8-bit AdamW           ~0.75 GB   (vs ~6 GB for fp32 Adam)
#   Activations  (w/ GC)  ~2–3 GB
#   ─────────────────────────────
#   Estimated total        ~9–10 GB / GPU  → safe on T4
#
# Techniques used (all from the project's own src/ package):
#   • DDP via accelerate.notebook_launcher     (2 GPUs, equal utilisation)
#   • bitsandbytes 8-bit AdamW                 (make_optimizer use_8bit=True)
#   • float16 AMP                              (T4 has fast fp16, no bf16)
#   • Gradient checkpointing                   (transformer use_checkpoint=True)
#   • Gradient accumulation × 16              (effective batch = 32)
#   • Depth-scaled residual init               (transformer._init_weights)
#   • RoPE with precomputed buffers            (MultiHeadAttention._apply_rope)
#   • SwiGLU feed-forward                      (FeedForward)
#   • Cosine LR schedule with linear warmup    (train.get_cosine_schedule_with_warmup)
#   • Unlikelihood loss                         (train._repetition_ul_loss)
#   • Step-based checkpointing                 (auto-resume across sessions)
# =============================================================================

## Cell 1 — Install dependencies
#
Run this cell once, then restart the kernel.

In [8]:
!pip install -q -U accelerate bitsandbytes datasets transformers huggingface_hub galore-torch
# Install the project package from GitHub (or set REPO_ROOT below instead):
!pip install -q git+https://github.com/sh20022002/small-Language-Model.git

# Verify that key packages are at mutually compatible versions
import subprocess as _sp
_r = _sp.run(['pip', 'show', 'datasets', 'huggingface_hub', 'accelerate'],
             capture_output=True, text=True)
for _line in _r.stdout.split('\n'):
    if _line.startswith(('Name:', 'Version:')):
        print(_line)

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Name: datasets
Version: 4.8.5
Name: huggingface_hub
Version: 1.17.0
Name: accelerate
Version: 1.13.0


## Cell 2 — sys.path setup + GPU + compute benchmark

In [9]:
import sys
from pathlib import Path

# ── REPO_ROOT: auto-detected from the Kaggle dataset path ────────────────────
# When pip-installed (Cell 1) the package is already on sys.path; REPO_ROOT stays None.
# When the repo is attached as a Kaggle dataset, the path is found automatically.
# Override manually if your dataset has a different slug.
_repo_candidate = "/kaggle/input/small-language-model"
REPO_ROOT = _repo_candidate if Path(_repo_candidate).is_dir() else None
if REPO_ROOT:
    for sub in ("src", "tests"):
        p = str(Path(REPO_ROOT) / sub)
        if p not in sys.path:
            sys.path.insert(0, p)
    print(f"[REPO_ROOT] {REPO_ROOT}")
else:
    print("[REPO_ROOT] Dataset not found — using pip-installed package.")

# ── GPU info via nvidia-smi (no torch.cuda — keeps CUDA uninitialised so
#    notebook_launcher can use fork/spawn without hitting the
#    "Cannot re-initialize CUDA in forked subprocess" error) ──────────────────
import subprocess

try:
    _smi = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True, text=True, timeout=10,
    )
    _gpu_lines = [l.strip() for l in _smi.stdout.strip().split("\n") if l.strip()]
    n_gpus = len(_gpu_lines)
    print(f"GPUs available: {n_gpus}")
    for i, line in enumerate(_gpu_lines):
        name, mem = line.split(",")
        print(f"  cuda:{i}  {name.strip()}  {float(mem.split()[0]) / 1024:.1f} GB")
except Exception as _e:
    print(f"[nvidia-smi] {_e}")
    n_gpus = 0

if n_gpus < 2:
    print("\nWARNING: fewer than 2 GPUs found — "
          "enable 'GPU T4 x2' in Kaggle Settings → Accelerator.")


# ── Ensure models/ directory exists (pre-trained stage checkpoints) ────────
# On Kaggle: upload your *.pt files as a Kaggle dataset, add it to this
# notebook, then set MODELS_DIR in Cell 3 to the dataset path, e.g.:
#   MODELS_DIR = "/kaggle/input/<your-dataset-name>/models"
_models_local = Path("models")
_models_local.mkdir(exist_ok=True)
print(f"[Models] Local models dir: {_models_local.resolve()}")
for _pt in sorted(_models_local.glob("*_stage.pt")):
    print(f"  found: {_pt.name}  ({_pt.stat().st_size / 1024**2:.0f} MB)")
if not any(_models_local.glob("*_stage.pt")):
    print("  (empty — will train from scratch or load from MODELS_DIR)")
# Compute benchmark — GPU GEMM TFLOPS + MFU (installed with the package)
try:
    from my_slm.mfu import run_perf_and_mfu
    if n_gpus >= 1:
        run_perf_and_mfu()
except ImportError:
    print("[mfu] Skipped — reinstall the package: pip install -q git+https://github.com/sh20022002/small-Language-Model.git")

GPUs available: 2
  cuda:0  Tesla T4  15.0 GB
  cuda:1  Tesla T4  15.0 GB
[mfu] Skipped — set REPO_ROOT above to enable, or pip-install the package.


## Cell 3 — Configuration
**Edit this cell only.** Everything else adapts automatically.

In [10]:
# ── Model configuration ────────────────────────────────────────────────────────
# Estimated parameter counts (GPT-2 tokenizer, vocab=50 257):
#   small  : dim=512,  depth=8,  heads=8,  mlp=2048   → ~110 M  ← default
#   medium : dim=768,  depth=12, heads=12, mlp=3072   → ~250 M
#   large  : dim=1024, depth=16, heads=16, mlp=4096   → ~650 M
MODEL_CFG = dict(
    dim     = 512,
    depth   = 8,
    heads   = 8,
    mlp_dim = 2048,
    window   = 2048,
    kv_heads = 2,       # GQA: 2 KV heads shared by 8 Q heads
    dropout  = 0.1,
)

# ── Tokenizer ──────────────────────────────────────────────────────────────────
TOKENIZER_PATH = None
BUILD_HYBRID   = False
HF_TOKENIZER   = "gpt2"

# ── Training stages (curriculum order) ────────────────────────────────────────
# Format: [name_or_names, steps, epochs, min_val_accuracy, max_epochs]
#
#   name_or_names : single dataset string OR list for a combined stage
#   steps > 0     : train exactly that many batches  (takes priority)
#   steps == 0    : run `epochs` base epochs, then accuracy gate up to max_epochs
#   min_val_accuracy: keep training past base epochs until this % val acc is reached
#   max_epochs    : hard cap; 0 = disable accuracy gate
#
# Curriculum:
#   Phase 1 — Pre-training   : simple text → factual → diverse web
#   Phase 2 — SFT            : instruction following, helpful conversations
#   Phase 3 — CoT / Reasoning: chain-of-thought, math reasoning
STAGES = [
    [["tinystories", "stories"],  0, 3, 28.0, 12],
    [["wikitext",    "c4"],          0, 3, 32.0, 12],
    [["openwebtext", "c4"],          0, 3, 33.0, 10],
    [["alpaca",      "dolly"],       0, 3, 38.0, 12],
    [["gsm8k",       "openorca"],    0, 3, 38.0, 12],
]
MAX_ITEMS_TRAIN = 20_000
MAX_ITEMS_VAL   =  2_000

# ── Training hyper-parameters ──────────────────────────────────────────────────
BATCH_SIZE    = 1        # per GPU; effective = 1 x 2 GPUs x 8 accum = 16
GRAD_ACCUM    = 8
LR            = 2e-4
WEIGHT_DECAY  = 0.1
USE_GALORE    = True   # GaLore gradient projection (saves ~50% optimizer memory)
GALORE_RANK   = 128    # projection rank; lower = more memory savings, less accuracy
GALORE_GAP    = 200    # steps between projection matrix updates
MAX_GRAD_NORM = 1.0
WARMUP_STEPS  = 300
UL_ALPHA      = 0.1

# ── Checkpointing ──────────────────────────────────────────────────────────────
OUTPUT_DIR       = "/kaggle/working/slm_run"
MODELS_DIR       = f"{REPO_ROOT}/models" if REPO_ROOT else "models"  # pre-trained stage checkpoints
SAVE_STEPS       = 500
SAVE_TOTAL_LIMIT = 3


## Cell 4 — Training function  (executed once per GPU via DDP)

In [11]:
def _train_fn():
    """
    Launched by notebook_launcher with num_processes=2.
    Each GPU process runs this function independently.
    All imports are local to avoid multiprocessing pickling issues.
    """
    import math, shutil
    from pathlib import Path

    import torch
    from accelerate import Accelerator
    from accelerate.utils import set_seed

    # ── Project imports ───────────────────────────────────────────────────────
    from my_slm.transformer import Transformer
    from my_slm.train import make_optimizer, get_cosine_schedule_with_warmup, load_latest_checkpoint
    from my_slm.multi_train_orchestrator import StageConfig, train_across_datasets
    from my_slm.hybrid_tokeniztion import HybridTokenizer

    # ── Accelerator ───────────────────────────────────────────────────────────
    accelerator = Accelerator(
        mixed_precision            = "fp16",   # T4: fp16 only (no bf16)
        gradient_accumulation_steps= GRAD_ACCUM,
    )
    set_seed(42)
    is_main = accelerator.is_main_process

    if is_main:
        Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
        print(f"[Accelerator] world={accelerator.num_processes}  "
              f"device={accelerator.device}  fp16=True  grad_accum={GRAD_ACCUM}")

    # ── Tokenizer ─────────────────────────────────────────────────────────────
    if TOKENIZER_PATH and Path(TOKENIZER_PATH).exists():
        tokenizer = HybridTokenizer.load(TOKENIZER_PATH)
        if is_main:
            print(f"[Tokenizer] HybridTokenizer loaded — vocab={tokenizer.vocab_size}")

    elif BUILD_HYBRID:
        from datasets import load_dataset
        tokenizer = HybridTokenizer()
        if is_main:
            print("[Tokenizer] Building HybridTokenizer from TinyStories …")
            stream = load_dataset("roneneldan/TinyStories",
                                  split="train", streaming=True)
            for i, ex in enumerate(stream):
                tokenizer.add_text(ex.get("text", ""))
                if i >= 20_000:
                    break
            tokenizer.freeze_vocab(k_bases=2000, max_merges=20_000)
            tok_out = Path(OUTPUT_DIR) / "tokenizer.pkl.gz"
            tokenizer.save(tok_out)
            print(f"[Tokenizer] Built — vocab={tokenizer.vocab_size}  → {tok_out}")
        accelerator.wait_for_everyone()
        if not is_main:
            tokenizer = HybridTokenizer.load(Path(OUTPUT_DIR) / "tokenizer.pkl.gz")

    else:
        from transformers import AutoTokenizer
        tokenizer = AutoTokenizer.from_pretrained(HF_TOKENIZER)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        if is_main:
            print(f"[Tokenizer] GPT-2 BPE — vocab={len(tokenizer)}")

    vocab_size = (tokenizer.vocab_size
                  if hasattr(tokenizer, "vocab_size") else len(tokenizer))

    # ── Model ─────────────────────────────────────────────────────────────────
    model = Transformer(
        vocab_size     = vocab_size,
        use_checkpoint = True,
        **MODEL_CFG,
    )
    if is_main:
        n = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"[Model] {n/1e6:.1f} M params ({n/1e9:.3f} B)  "
              f"dim={MODEL_CFG['dim']}  depth={MODEL_CFG['depth']}  "
              f"heads={MODEL_CFG['heads']}")

    # ── Architecture sanity check (pre-training, main process only) ───────────
    if is_main:
        try:
            # tests/test_model.py — notebook-callable entry point
            from test_model import check_model_architecture
            ok = check_model_architecture(model, vocab_size, device="cpu")
            if not ok:
                raise RuntimeError("Architecture checks failed — fix before training!")
        except ImportError:
            print("[check] test_model.py not found — set REPO_ROOT to enable.")

    # ── Optimizer (8-bit AdamW from bitsandbytes) ─────────────────────────────
    optimizer = make_optimizer(
        model,
        lr                   = LR,
        weight_decay         = WEIGHT_DECAY,
        betas                = (0.9, 0.95),
        use_8bit             = True,
        use_galore           = USE_GALORE,
        galore_rank          = GALORE_RANK,
        galore_update_proj_gap = GALORE_GAP,
    )

    # ── LR scheduler ─────────────────────────────────────────────────────────
    total_opt_steps = sum(
        (math.ceil(steps / GRAD_ACCUM) if steps > 0
        else math.ceil(MAX_ITEMS_TRAIN / max(BATCH_SIZE, 1) * max(max_ep, epochs, 1) / GRAD_ACCUM))
        for _, steps, epochs, _acc, max_ep in STAGES
    )
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        warmup_steps = WARMUP_STEPS,
        total_steps  = total_opt_steps,
    )

    # ── Prepare with Accelerator ──────────────────────────────────────────────
    model, optimizer, scheduler = accelerator.prepare(model, optimizer, scheduler)

    # ── Checkpoint helpers ────────────────────────────────────────────────────
    def _save(step: int):
        if not is_main:
            return
        ckpt_dir = Path(OUTPUT_DIR) / f"checkpoint-{step}"
        ckpt_dir.mkdir(parents=True, exist_ok=True)
        torch.save({
            "config": {**MODEL_CFG, "vocab_size": vocab_size},
            "model_state": accelerator.unwrap_model(model).state_dict(),
            "optimizer":   optimizer.state_dict(),
            "step":        step,
        }, ckpt_dir / "state.pt")
        # Enforce SAVE_TOTAL_LIMIT
        ckpts = sorted(
            [d for d in Path(OUTPUT_DIR).iterdir()
             if d.is_dir() and d.name.startswith("checkpoint-")],
            key=lambda d: int(d.name.split("-")[1]),
        )
        for old in ckpts[:-SAVE_TOTAL_LIMIT]:
            shutil.rmtree(old)
        print(f"[Checkpoint] step={step}  → {ckpt_dir}")

    global_step = load_latest_checkpoint(
        model, OUTPUT_DIR,
        models_dir=MODELS_DIR,
        optimizer=optimizer,
        accelerator=accelerator,
    )
    if is_main and not global_step:
        print("[Checkpoint] No checkpoint found — training from scratch.")

    # ── Multi-stage curriculum training (via orchestrator) ────────────────────
    stages = [
        StageConfig(name=name, steps=steps, epochs=epochs,
                    min_val_accuracy=min_acc, max_epochs=max_ep)
        for name, steps, epochs, min_acc, max_ep in STAGES
    ]

    model = train_across_datasets(
        model              = model,
        optimizer          = optimizer,
        tokenizer          = tokenizer,
        accelerator        = accelerator,
        stages             = stages,
        max_len            = MODEL_CFG["window"],
        train_items        = MAX_ITEMS_TRAIN,
        val_items          = MAX_ITEMS_VAL,
        batch_size         = BATCH_SIZE,
        scheduler          = scheduler,
        max_grad_norm      = MAX_GRAD_NORM,
        ul_alpha           = UL_ALPHA,
        save_dir           = OUTPUT_DIR,
    )

    global_step += sum(
        (math.ceil(steps / GRAD_ACCUM) if steps > 0
        else math.ceil(MAX_ITEMS_TRAIN / max(BATCH_SIZE, 1) * max(max_ep, epochs, 1) / GRAD_ACCUM))
        for _, steps, epochs, _acc, max_ep in STAGES
    )
    _save(global_step)

    # ── Final weights ─────────────────────────────────────────────────────────
    accelerator.wait_for_everyone()
    if is_main:
        final_path = Path(OUTPUT_DIR) / "final_model.pt"
        torch.save({
            "config": {**MODEL_CFG, "vocab_size": vocab_size},
            "model_state": accelerator.unwrap_model(model).state_dict(),
        }, final_path)
        print(f"\n[Done] Final weights → {final_path}")

    accelerator.end_training()

## Cell 5 — Launch DDP (2 processes, one per T4)

In [12]:
# torchrun uses spawn (fresh processes) — no CUDA fork issues.
# sys.executable ensures the worker uses the SAME Python / site-packages
# as this Jupyter kernel, so pip-installed packages are always visible.
import inspect, json, os, subprocess, sys
from pathlib import Path

# ── Serialise all notebook config vars to disk ─────────────────────────
Path('/tmp/slm_cfg.json').write_text(json.dumps(dict(
    MODEL_CFG=MODEL_CFG, STAGES=STAGES,
    MAX_ITEMS_TRAIN=MAX_ITEMS_TRAIN, MAX_ITEMS_VAL=MAX_ITEMS_VAL,
    BATCH_SIZE=BATCH_SIZE, GRAD_ACCUM=GRAD_ACCUM, LR=LR,
    WEIGHT_DECAY=WEIGHT_DECAY, MAX_GRAD_NORM=MAX_GRAD_NORM,
    USE_GALORE=USE_GALORE, GALORE_RANK=GALORE_RANK, GALORE_GAP=GALORE_GAP,
    MODELS_DIR=MODELS_DIR,
    WARMUP_STEPS=WARMUP_STEPS, UL_ALPHA=UL_ALPHA,
    OUTPUT_DIR=OUTPUT_DIR, SAVE_STEPS=SAVE_STEPS,
    SAVE_TOTAL_LIMIT=SAVE_TOTAL_LIMIT,
    TOKENIZER_PATH=TOKENIZER_PATH, BUILD_HYBRID=BUILD_HYBRID,
    HF_TOKENIZER=HF_TOKENIZER, REPO_ROOT=REPO_ROOT,
)))

# ── Build worker script ───────────────────────────────────────────────────────────────────────
_preamble = [
    'import json, sys',
    'from pathlib import Path',
    'cfg = json.loads(Path("/tmp/slm_cfg.json").read_text())',
    'for k, v in cfg.items(): globals()[k] = v',
    'if globals().get("REPO_ROOT"):',
    '    for s in ("src", "tests"):',
    '        p = str(Path(globals()["REPO_ROOT"]) / s)',
    '        if p not in sys.path: sys.path.insert(0, p)',
    '']
_fn_src = inspect.getsource(_train_fn)
_worker_src = '\n'.join(_preamble) + _fn_src + '\nif __name__ == "__main__": _train_fn()\n'
Path('/tmp/slm_worker.py').write_text(_worker_src)

# ── Verify worker script ───────────────────────────────────────────────────────────────────────
print("=== Worker script (first 20 lines) ===")
for _i, _ln in enumerate(_worker_src.split('\n')[:20]):
    print(f"{_i+1:3d}  {_ln}")
print()

_chk = subprocess.run([sys.executable, '-m', 'py_compile', '/tmp/slm_worker.py'],capture_output=True, text=True)
if _chk.returncode != 0:
    raise SyntaxError(f"Worker script syntax error:{_chk.stderr}")
print("[OK] Syntax check passed")
# ── Launch via torch.distributed.run (same Python as this kernel) ──────────────────
# Using sys.executable guarantees the worker processes inherit the exact same
# Python interpreter and site-packages as this Jupyter kernel, so every
# pip-installed package (datasets, huggingface_hub, my_slm, ...) is available.
_cmd = (
    f'{sys.executable} -m torch.distributed.run'
    f' --standalone --nproc_per_node={n_gpus or 2}'
    ' --master_port=29500'
    ' /tmp/slm_worker.py')
print(f"Running: {_cmd}")
_ret = os.system(_cmd)
if _ret != 0:
    raise RuntimeError(f'torchrun exited with code {_ret}')

# ── Display saved loss-curve PNGs in the notebook ───────────────────────────
import matplotlib.pyplot as plt
import matplotlib.image as _mpimg
from pathlib import Path as _Path

_figs = sorted(_Path(OUTPUT_DIR).glob("*_loss.png"))
if _figs:
    _cols = min(3, len(_figs))
    _rows = (_len := len(_figs), (_len + _cols - 1) // _cols)[1]
    _fig, _axes = plt.subplots(_rows, _cols, figsize=(5 * _cols, 4 * _rows),
                               squeeze=False)
    for _ax in _axes.flat:
        _ax.axis("off")
    for _ax, _fp in zip(_axes.flat, _figs):
        _ax.imshow(_mpimg.imread(str(_fp)))
        _ax.set_title(_fp.stem.replace("_loss", ""), fontsize=10)
        _ax.axis("off")
    plt.suptitle("Per-stage training loss", fontsize=12)
    plt.tight_layout()
    plt.show()
    print(f"[Plots] Displayed {len(_figs)} loss curves from {OUTPUT_DIR}")
else:
    print("[Plots] No *_loss.png files found — run training first.")


=== Worker script (first 20 lines) ===
  1  import json, sys
  2  from pathlib import Path
  3  cfg = json.loads(Path("/tmp/slm_cfg.json").read_text())
  4  for k, v in cfg.items(): globals()[k] = v
  5  if globals().get("REPO_ROOT"):
  6      for s in ("src", "tests"):
  7          p = str(Path(globals()["REPO_ROOT"]) / s)
  8          if p not in sys.path: sys.path.insert(0, p)
  9  def _train_fn():
 10      """
 11      Launched by notebook_launcher with num_processes=2.
 12      Each GPU process runs this function independently.
 13      All imports are local to avoid multiprocessing pickling issues.
 14      """
 15      import math, shutil
 16      from pathlib import Path
 17  
 18      import torch
 19      from accelerate import Accelerator
 20      from accelerate.utils import set_seed

[OK] Syntax check passed
Running: /usr/bin/python3 -m torch.distributed.run --standalone --nproc_per_node=2 --master_port=29500 /tmp/slm_worker.py



*****************************************
Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
*****************************************
[W528 18:58:33.026967659 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W528 18:58:37.294433892 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W528 18:58:37.471849817 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


[Accelerator] world=2  device=cuda:0  fp16=True  grad_accum=16


[Tokenizer] GPT-2 BPE — vocab=50257
[Model] 50.9 M params (0.051 B)  dim=512  depth=8  heads=8
[check] test_model.py not found — set REPO_ROOT to enable.

=== Stage: tinystories | epochs=2 | steps=0 ===
Started Training (Accelerate / DDP) …
Epoch 1/2 — Train Loss: nan, Val Loss: 2.2350, Acc: 50.63%
Epoch 2/2 — Train Loss: nan, Val Loss: 2.0056, Acc: 53.87%
Figure(600x400)
[Checkpoint] Saved /kaggle/working/slm_run/tinystories_stage.pt

=== Stage: wikitext | epochs=2 | steps=0 ===
Started Training (Accelerate / DDP) …
Epoch 1/2 — Train Loss: nan, Val Loss: nan, Acc: 23.59%
Epoch 2/2 — Train Loss: nan, Val Loss: nan, Acc: 25.31%
Figure(600x400)
[Checkpoint] Saved /kaggle/working/slm_run/wikitext_stage.pt

=== Stage: openwebtext | epochs=2 | steps=0 ===
Started Training (Accelerate / DDP) …
Epoch 1/2 — Train Loss: 5.1963, Val Loss: 4.7531, Acc: 25.68%
Epoch 2/2 — Train Loss: 4.7381, Val Loss: 4.5113, Acc: 27.80%
Figure(600x400)
[Checkpoint] Saved /kaggle/working/slm_run/openwebtext_stage.

## Cell 5b — Resume training from last checkpoint
Run this cell (instead of Cell 5) after a Kaggle timeout, crash, or mid-stage interruption.  
Set `RESUME_STAGE` to a stage index (0-based) to force a specific restart point, or leave `None` for auto-detection.

In [ ]:
# ── Resume training from last checkpoint ───────────────────────────────────────────────────────────────────
#
# RESUME_STAGE : None  → auto-detect (first stage without a saved *_stage.pt file)
#                int   → force restart from that stage index  (0 = first stage)
#
RESUME_STAGE = None

import inspect, json, os, subprocess, sys
from pathlib import Path

# ── Write full config to disk (same as launch cell + RESUME_STAGE) ────────────────
Path("/tmp/slm_cfg.json").write_text(json.dumps(dict(
    MODEL_CFG=MODEL_CFG, STAGES=STAGES,
    MAX_ITEMS_TRAIN=MAX_ITEMS_TRAIN, MAX_ITEMS_VAL=MAX_ITEMS_VAL,
    BATCH_SIZE=BATCH_SIZE, GRAD_ACCUM=GRAD_ACCUM, LR=LR,
    WEIGHT_DECAY=WEIGHT_DECAY, MAX_GRAD_NORM=MAX_GRAD_NORM,
    WARMUP_STEPS=WARMUP_STEPS, UL_ALPHA=UL_ALPHA,
    OUTPUT_DIR=OUTPUT_DIR, SAVE_STEPS=SAVE_STEPS,
    SAVE_TOTAL_LIMIT=SAVE_TOTAL_LIMIT,
    TOKENIZER_PATH=TOKENIZER_PATH, BUILD_HYBRID=BUILD_HYBRID,
    HF_TOKENIZER=HF_TOKENIZER, REPO_ROOT=REPO_ROOT,
    USE_GALORE=USE_GALORE, GALORE_RANK=GALORE_RANK, GALORE_GAP=GALORE_GAP,
    MODELS_DIR=MODELS_DIR,
    RESUME_STAGE=RESUME_STAGE,
)))

# ── Worker function ───────────────────────────────────────────────────────────────────
def _resume_fn():
    import math, shutil
    from pathlib import Path
    import torch
    from accelerate import Accelerator
    from accelerate.utils import set_seed
    from my_slm.transformer import Transformer
    from my_slm.train import make_optimizer, get_cosine_schedule_with_warmup, load_latest_checkpoint
    from my_slm.multi_train_orchestrator import StageConfig, train_across_datasets

    accelerator = Accelerator(
        mixed_precision=             "fp16",
        gradient_accumulation_steps= GRAD_ACCUM,
    )
    set_seed(42)
    is_main = accelerator.is_main_process

    if is_main:
        Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
        print(f"[Resume] world={accelerator.num_processes}  device={accelerator.device}")

    # ── Tokenizer ───────────────────────────────────────────────────────────────
    if TOKENIZER_PATH and Path(TOKENIZER_PATH).exists():
        from my_slm.hybrid_tokeniztion import HybridTokenizer
        tokenizer = HybridTokenizer.load(TOKENIZER_PATH)
        if is_main:
            print(f"[Tokenizer] HybridTokenizer — vocab={tokenizer.vocab_size}")
    else:
        from transformers import AutoTokenizer
        tokenizer = AutoTokenizer.from_pretrained(HF_TOKENIZER)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        if is_main:
            print(f"[Tokenizer] {HF_TOKENIZER} BPE — vocab={len(tokenizer)}")
    vocab_size = getattr(tokenizer, "vocab_size", len(tokenizer))

    # ── Model ───────────────────────────────────────────────────────────────────────
    model = Transformer(vocab_size=vocab_size, use_checkpoint=True, **MODEL_CFG)
    if is_main:
        n = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"[Model] {n/1e6:.1f} M params")

    # ── Optimizer + scheduler (same total budget as original run) ────────────────────
    optimizer = make_optimizer(
        model,
        lr=LR, weight_decay=WEIGHT_DECAY, betas=(0.9, 0.95),
        use_8bit=True,
        use_galore=USE_GALORE, galore_rank=GALORE_RANK,
        galore_update_proj_gap=GALORE_GAP,
    )
    total_opt_steps = sum(
        (math.ceil(steps / GRAD_ACCUM) if steps > 0
         else math.ceil(MAX_ITEMS_TRAIN / max(BATCH_SIZE, 1)
                        * max(max_ep, epochs, 1) / GRAD_ACCUM))
        for _, steps, epochs, _acc, max_ep in STAGES
    )
    scheduler = get_cosine_schedule_with_warmup(
        optimizer, warmup_steps=WARMUP_STEPS, total_steps=total_opt_steps,
    )
    model, optimizer, scheduler = accelerator.prepare(model, optimizer, scheduler)

    # ── Load checkpoint (step ckpt → stage ckpt → pre-trained models_dir) ─────────
    global_step = load_latest_checkpoint(
        model, OUTPUT_DIR,
        models_dir=MODELS_DIR,
        optimizer=optimizer,
        accelerator=accelerator,
    )
    if is_main and not global_step:
        print("[Resume] No checkpoint found — starting from scratch.")

    # ── Detect which stages remain ──────────────────────────────────────────────────
    all_stages = [
        StageConfig(name=name, steps=steps, epochs=epochs,
                    min_val_accuracy=min_acc, max_epochs=max_ep)
        for name, steps, epochs, min_acc, max_ep in STAGES
    ]

    def _safe(s):
        ns = s.name if isinstance(s.name, list) else [s.name]
        return "+".join(n.lower() for n in ns).replace("+", "_")

    start = RESUME_STAGE
    if start is None:
        start = 0
        for i, s in enumerate(all_stages):
            if (Path(OUTPUT_DIR) / f"{_safe(s)}_stage.pt").exists():
                start = i + 1

    if is_main:
        done = [_safe(s) for s in all_stages[:start]]
        todo = [_safe(s) for s in all_stages[start:]]
        if done:
            print(f"[Resume] Completed : {done}")
        print(f"[Resume] Remaining : {todo if todo else '(nothing — all done)'}")

    stages_to_run = all_stages[start:]
    if not stages_to_run:
        if is_main:
            print("[Resume] All stages already complete.")
        accelerator.end_training()
        return

    # ── Train remaining stages ────────────────────────────────────────────────────────
    model = train_across_datasets(
        model=model, optimizer=optimizer, tokenizer=tokenizer,
        accelerator=accelerator,
        stages=stages_to_run,
        max_len=MODEL_CFG["window"],
        train_items=MAX_ITEMS_TRAIN, val_items=MAX_ITEMS_VAL,
        batch_size=BATCH_SIZE,
        scheduler=scheduler,
        max_grad_norm=MAX_GRAD_NORM, ul_alpha=UL_ALPHA,
        save_dir=OUTPUT_DIR,
    )

    # ── Save checkpoint + inference weights ──────────────────────────────────────
    global_step += sum(
        (math.ceil(steps / GRAD_ACCUM) if steps > 0
         else math.ceil(MAX_ITEMS_TRAIN / max(BATCH_SIZE, 1)
                        * max(max_ep, epochs, 1) / GRAD_ACCUM))
        for _, steps, epochs, _acc, max_ep in stages_to_run
    )
    accelerator.wait_for_everyone()
    if is_main:
        _ckpt_dir = Path(OUTPUT_DIR) / f"checkpoint-{global_step}"
        _ckpt_dir.mkdir(parents=True, exist_ok=True)
        torch.save({
            "config":      {**MODEL_CFG, "vocab_size": vocab_size},
            "model_state": accelerator.unwrap_model(model).state_dict(),
            "optimizer":   optimizer.state_dict(),
            "step":        global_step,
        }, _ckpt_dir / "state.pt")
        _all = sorted(
            [d for d in Path(OUTPUT_DIR).iterdir()
             if d.is_dir() and d.name.startswith("checkpoint-")],
            key=lambda d: int(d.name.split("-")[1]),
        )
        for _old in _all[:-SAVE_TOTAL_LIMIT]:
            shutil.rmtree(_old)
        print(f"[Checkpoint] step={global_step}  → {_ckpt_dir}")

        _final = Path(OUTPUT_DIR) / "final_model.pt"
        torch.save({
            "config":      {**MODEL_CFG, "vocab_size": vocab_size},
            "model_state": accelerator.unwrap_model(model).state_dict(),
        }, _final)
        print(f"[Done] Final weights → {_final}")

    accelerator.end_training()

# ── Build worker script + syntax check ───────────────────────────────────────────────
_preamble = [
    "import json, sys",
    "from pathlib import Path",
    "cfg = json.loads(Path('/tmp/slm_cfg.json').read_text())",
    "for k, v in cfg.items(): globals()[k] = v",
    "if globals().get('REPO_ROOT'):",
    "    for s in ('src', 'tests'):",
    "        p = str(Path(globals()['REPO_ROOT']) / s)",
    "        if p not in sys.path: sys.path.insert(0, p)",
    "",
]
_NL = chr(10)
_worker = (_NL.join(_preamble)
           + inspect.getsource(_resume_fn)
           + _NL + "if __name__ == '__main__': _resume_fn()" + _NL)
Path("/tmp/slm_resume_worker.py").write_text(_worker)

_chk = subprocess.run([sys.executable, "-m", "py_compile", "/tmp/slm_resume_worker.py"],
                      capture_output=True, text=True)
if _chk.returncode != 0:
    raise SyntaxError(f"Resume worker syntax error:{_chk.stderr}")
print("[OK] Resume worker syntax check passed")

# ── Launch via torchrun (port 29501 avoids clash with main launch cell) ───────────
_n   = n_gpus if "n_gpus" in dir() else 2
_cmd = (f"{sys.executable} -m torch.distributed.run"
        f" --standalone --nproc_per_node={_n}"
        f" --master_port=29501"
        f" /tmp/slm_resume_worker.py")
print(f"Running: {_cmd}")
_ret = os.system(_cmd)
if _ret != 0:
    raise RuntimeError(f"torchrun exited with code {_ret}")

# ── Display all loss curves ──────────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.image as _mpimg

_figs = sorted(Path(OUTPUT_DIR).glob("*_loss.png"))
if _figs:
    _cols = min(3, len(_figs))
    _rows = (len(_figs) + _cols - 1) // _cols
    _fig, _axes = plt.subplots(_rows, _cols, figsize=(5 * _cols, 4 * _rows), squeeze=False)
    for _ax in _axes.flat: _ax.axis("off")
    for _ax, _fp in zip(_axes.flat, _figs):
        _ax.imshow(_mpimg.imread(str(_fp)))
        _ax.set_title(_fp.stem.replace("_loss", ""), fontsize=10)
        _ax.axis("off")
    plt.suptitle("Per-stage training loss", fontsize=12)
    plt.tight_layout()
    plt.show()
    print(f"[Plots] {len(_figs)} loss curve(s) from {OUTPUT_DIR}")
else:
    print("[Plots] No *_loss.png files found yet.")


## Cell 6 — Post-training checks + semantic evaluation
#
Run after Cell 5 completes.  Requires REPO_ROOT set in Cell 2.

In [16]:
def _post_training_eval(quick: bool = True):
    """
    1. Architecture + training behaviour checks (test_model, test_training).
    2. Semantic benchmarks (perplexity, top-k, BLiMP, LAMBADA, analogy).
    """
    from pathlib import Path
    import torch

    final_path = Path(OUTPUT_DIR) / "final_model.pt"
    if not final_path.exists():
        print(f"[Eval] {final_path} not found — run Cell 5 first.")
        return

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Load model + tokenizer via semantic_eval helper
    try:
        from my_slm.semantic_eval import load_model_and_tok, run_all, print_report
    except ImportError:
        print("[Eval] semantic_eval.py not found — set REPO_ROOT in Cell 2.")
        return

    tok_src = TOKENIZER_PATH or HF_TOKENIZER
    model, tok = load_model_and_tok(str(final_path), tok_src, device)

    # ── Architecture checks ───────────────────────────────────────────────────
    try:
        from test_model import check_model_architecture
        check_model_architecture(model, model.token_emb.num_embeddings, device)
    except ImportError:
        pass

    # ── Post-training behaviour checks ────────────────────────────────────────
    try:
        from test_training import check_trained_model
        from my_slm.multi_train_orchestrator import _get_pad_id
        check_trained_model(model, tok, device,
                            vocab_size=model.token_emb.num_embeddings,
                            pad_id=_get_pad_id(tok))
    except ImportError:
        pass

    # ── Semantic benchmarks ───────────────────────────────────────────────────
    report = run_all(model, tok, device, quick=quick)
    print_report(report)
    return report


# Uncomment to run:
_post_training_eval(quick=True)

[Eval] semantic_eval.py not found — set REPO_ROOT in Cell 2.


## Cell 7 — Generation test

In [19]:
def _generate(prompt: str, max_new_tokens: int = 120, temperature: float = 0.8):
    from pathlib import Path
    import torch
    from my_slm.transformer import Transformer
    from my_slm.multi_train_orchestrator import _encode, _get_pad_id

    final_path = Path(OUTPUT_DIR) / "final_model.pt"
    ckpt = torch.load(final_path, map_location="cpu")

    cfg        = ckpt["config"]
    state_dict = ckpt["model_state"]

    if TOKENIZER_PATH and Path(TOKENIZER_PATH).exists():
        from my_slm.hybrid_tokeniztion import HybridTokenizer
        tok = HybridTokenizer.load(TOKENIZER_PATH)
    else:
        from transformers import AutoTokenizer
        tok = AutoTokenizer.from_pretrained(HF_TOKENIZER)
        if tok.pad_token is None:
            tok.pad_token = tok.eos_token

    _keys = ("vocab_size", "dim", "depth", "heads", "mlp_dim", "window")
    model = Transformer(**{k: cfg[k] for k in _keys if k in cfg},
                        kv_heads=cfg.get("kv_heads"),
                        dropout=0.0, use_checkpoint=False)
    model.load_state_dict(state_dict, strict=False)
    model.eval()

    eos_id = _get_pad_id(tok)
    ids    = torch.tensor([_encode(tok, prompt)], dtype=torch.long)

    with torch.inference_mode():
        out = model.generate(
            ids,
            max_new_tokens     = max_new_tokens,
            temperature        = temperature,
            top_k              = 50,
            eos_token_id       = eos_id,   # stop as soon as EOS is produced
            repetition_penalty = 1.3,
        )

    gen_ids = out[0, len(_encode(tok, prompt)):].tolist()
    if hasattr(tok, "token2id"):
        return tok.decode(gen_ids)
    return tok.decode(gen_ids, skip_special_tokens=True)


print(_generate("what is the capital of france?"))   # uncomment to test


It has been built, but no one ever saw it. In return for its construction and design were the main thing we had in years to explore!
A few words which have been completed to make use of ancient art are: 
The capital was once named after its owner, John, who said that there is no place to be built on buildings such as Mount Huxab. He believed he had a lot left with just one thing at home when you needed it not before - as if his actions were made in a big city or near country where the city is located today and then


In [15]:
_post_training_eval()

[Eval] semantic_eval.py not found — set REPO_ROOT in Cell 2.
